# S6E9 | Multi-Resolution Nested Target Encoding + Partition-Bagged LightGBM

Playground Series S6E9 is predicting `Will_Buy_EV` from ~14 demographic and behavioural columns. It's synthetic data, and once you notice that, the whole approach to this competition changes: you're not modelling car-buying behaviour, you're modelling whatever script generated these particular 668,665 rows.

This notebook is built around one idea that ended up carrying almost the entire gap to the top of the board: the two high-cardinality numeric columns (`Annual_Income_USD`, `Daily_Commute_km`) hold per-value structure that a gradient-boosted tree structurally cannot reach on its own, no matter how much you tune it. Everything below either exploits that or checks whether something else does too.

Scores on the frozen split I used everywhere, `StratifiedKFold(5, shuffle=True, random_state=42)`:

| step | OOF AUC | Public LB |
|---|---|---|
| raw features, plain XGBoost | 0.94176 | – |
| + nested target encoding (income, commute) | 0.94512 | – |
| + frequency encoding | 0.94544 | 0.94574 |
| + LightGBM, more capacity | 0.94563 | 0.94592 |
| + multi-resolution keys, triple TE | 0.94608 | 0.94635 |
| + partition bagging (final) | 0.94631 | **0.94638** |

I'll go through the reasoning for each step, then a section on what I tried that *didn't* move the score — there's more of that than there is of the stuff that worked, and it's worth knowing about before you spend a submission re-discovering it.

## 1. Setup

In [ ]:
import json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import issparse

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

import lightgbm as lgb

SEED = 42
N_SPLITS = 5
TARGET = "Will_Buy_EV"
ID_COL = "id"

def resolve_input_dir():
    candidates = [
        Path("/kaggle/input/playground-series-s6e9"),
        Path("/kaggle/input/competitions/playground-series-s6e9"),
    ]
    for c in candidates:
        if (c / "train.csv").exists():
            return c
    raise FileNotFoundError(f"train.csv not found under any of: {candidates}")

INPUT_DIR = resolve_input_dir()
train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")
sample = pd.read_csv(INPUT_DIR / "sample_submission.csv")

y = train[TARGET].map({"Yes": 1, "No": 0}).astype(int)
X = train.drop(columns=[TARGET, ID_COL])
X_test = test.drop(columns=[ID_COL])

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print(f"train: {train.shape}, test: {test.shape}")
print(f"positive rate: {y.mean():.4f}")
print(f"numeric: {num_cols}")
print(f"categorical: {cat_cols}")

## 2. Why the two numeric columns matter more than they look like they should

The first thing I check on any Playground episode is cardinality — how many distinct values each column actually holds, versus how many rows there are.

In [ ]:
both = pd.concat([X, X_test])
card = {c: both[c].nunique() for c in X.columns}
for c, n in sorted(card.items(), key=lambda kv: -kv[1]):
    print(f"{c:32s} {n:>7,d} distinct")

In [ ]:
print("top income values by count:")
print(X["Annual_Income_USD"].value_counts().head(5))
print()
print("top commute values by count:")
print(X["Daily_Commute_km"].value_counts().head(5))
print()

commute_spike = X["Daily_Commute_km"] == 5.0
rate_spike = y[commute_spike].mean()
rate_rest = y[~commute_spike].mean()
print(f"commute == 5.0 km: {commute_spike.sum():,} rows ({commute_spike.mean():.1%} of train), "
      f"buy rate {rate_spike:.4f} vs {rate_rest:.4f} everywhere else")

long_commute = X["Daily_Commute_km"] >= 83
print(f"commute >= 83 km: {long_commute.sum()} rows, buy rate {y[long_commute].mean():.4f}")

`Annual_Income_USD` and `Daily_Commute_km` are in a completely different league from everything else — tens of thousands of distinct values against single or double digits for the rest. And they're not smoothly distributed either: the counts above show a huge spike sitting right at an income of 30000, and an even bigger one on commute at exactly 5.0 km, buying at a visibly higher rate than the rest of the file. There's also a hard floor at the other end — past a certain commute distance, EV purchases stop appearing in the data entirely.

None of that is a coincidence and none of it is about EV buyers. It's what you get when a generator samples 668k rows from a fitted distribution that has both a continuous component and some fixed spike values baked in. The catch is that a tree splits on *ranges*: isolating one spike value out of 13,000+ distinct income values costs two splits (one to bracket it, one to peel it off), and you don't have the depth budget to do that for hundreds of spikes. Whatever the model is going to learn about `Annual_Income_USD`, it's mostly not going to be the per-value structure — unless you hand that structure to it as a column.

## 3. Baseline: raw features, one-hot categoricals, plain XGBoost

Before doing anything clever, get an honest number for what "nothing clever" scores. Median-impute the numerics, one-hot the categoricals, drop `id`, and fit a single boosted model.

In [ ]:
from xgboost import XGBClassifier

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", ohe)]), cat_cols),
])

X_all = preprocess.fit_transform(X)
X_test_all = preprocess.transform(X_test)
if issparse(X_all):
    X_all, X_test_all = X_all.toarray(), X_test_all.toarray()
X_all = np.ascontiguousarray(X_all, dtype=np.float32)
X_test_all = np.ascontiguousarray(X_test_all, dtype=np.float32)

folds = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(X_all, y))

oof_baseline = np.zeros(len(train))
for tr_idx, va_idx in folds:
    model = XGBClassifier(n_estimators=2000, max_depth=6, learning_rate=0.05,
                           tree_method="hist", eval_metric="auc",
                           early_stopping_rounds=100, random_state=SEED)
    model.fit(X_all[tr_idx], y.iloc[tr_idx], eval_set=[(X_all[va_idx], y.iloc[va_idx])], verbose=False)
    oof_baseline[va_idx] = model.predict_proba(X_all[va_idx])[:, 1]

print(f"raw-feature baseline OOF AUC: {roc_auc_score(y, oof_baseline):.5f}")

0.9418 on the nose. That's the floor — everything from here is about closing the gap to the high 0.946s people are reporting on the board, and almost none of that gap turns out to be about model choice or tuning.

## 4. The main idea: nested per-value target encoding

If a tree can't isolate individual income/commute values, hand it a column that already carries each value's own target rate. The one rule that matters here is that a row can never see a statistic computed using its own label, or the encoding leaks and cross-validation lies to you.

I do this with a two-level nested split: for each outer fold, the *validation* rows get a target rate fitted on the whole outer-training partition, but the *training* rows themselves get a rate fitted on an **inner** 5-fold split, so no training row ever contributes to its own encoding. I also run a leakage self-test before trusting any of this: encode a column of pure random noise the nested way and the leaky way. The nested version should score essentially 0.5 (it knows nothing); the leaky version should score well above it. If that check ever fails, stop and find the bug before reading anything else on the page.

In [ ]:
TE_COLS = ["Annual_Income_USD", "Daily_Commute_km"]
TE_SMOOTHING = 10.0
TE_INNER_SPLITS = 5

def fit_target_rate(values, labels):
    agg = labels.groupby(values.to_numpy()).agg(["sum", "count"])
    return agg["sum"], agg["count"], float(labels.mean())

def apply_target_rate(values, sums, counts, prior, smoothing=TE_SMOOTHING):
    v = pd.Series(values.to_numpy())
    s = v.map(sums).fillna(0.0).to_numpy(dtype=np.float64)
    c = v.map(counts).fillna(0.0).to_numpy(dtype=np.float64)
    return (s + prior * smoothing) / (c + smoothing)

def nested_target_encode(cols, tr_idx, va_idx, fold_seed, leaky=False, smoothing=TE_SMOOTHING):
    te_tr = np.zeros((len(tr_idx), len(cols)))
    te_va = np.zeros((len(va_idx), len(cols)))
    te_te = np.zeros((len(X_test), len(cols)))
    y_tr = y.iloc[tr_idx]
    for j, col in enumerate(cols):
        v_tr = X[col].iloc[tr_idx]
        sums, counts, prior = fit_target_rate(v_tr, y_tr)
        te_va[:, j] = apply_target_rate(X[col].iloc[va_idx], sums, counts, prior, smoothing)
        te_te[:, j] = apply_target_rate(X_test[col], sums, counts, prior, smoothing)
        if leaky:
            te_tr[:, j] = apply_target_rate(v_tr, sums, counts, prior, smoothing)
            continue
        inner = StratifiedKFold(TE_INNER_SPLITS, shuffle=True, random_state=fold_seed)
        for i_tr, i_va in inner.split(v_tr, y_tr):
            s_i, c_i, p_i = fit_target_rate(v_tr.iloc[i_tr], y_tr.iloc[i_tr])
            te_tr[i_va, j] = apply_target_rate(v_tr.iloc[i_va], s_i, c_i, p_i, smoothing)
    return te_tr, te_va, te_te

# Leakage self-test. A pure-noise column can only score above chance if the
# encoder is somehow reading a row's own label back into its own feature, so
# this is a direct check on the mechanism rather than on any real column. The
# noise needs roughly the same cardinality as the real high-cardinality
# columns to be a fair stand-in -- a low-cardinality noise column has so much
# support per value that even the leaky version barely leaks.
rng = np.random.default_rng(SEED)
noise_card = X["Annual_Income_USD"].nunique()
X_probe = X.assign(__noise__=rng.integers(0, noise_card, len(X)))
tr0, va0 = folds[0]

def _probe_encode(tr_idx, va_idx, leaky):
    y_tr = y.iloc[tr_idx]
    v_tr = X_probe["__noise__"].iloc[tr_idx]
    sums, counts, prior = fit_target_rate(v_tr, y_tr)
    if leaky:
        return apply_target_rate(v_tr, sums, counts, prior)
    te_tr = np.zeros(len(tr_idx))
    inner = StratifiedKFold(TE_INNER_SPLITS, shuffle=True, random_state=SEED + 100)
    for i_tr, i_va in inner.split(v_tr, y_tr):
        s_i, c_i, p_i = fit_target_rate(v_tr.iloc[i_tr], y_tr.iloc[i_tr])
        te_tr[i_va] = apply_target_rate(v_tr.iloc[i_va], s_i, c_i, p_i)
    return te_tr

auc_nested = roc_auc_score(y.iloc[tr0], _probe_encode(tr0, va0, leaky=False))
auc_leaky = roc_auc_score(y.iloc[tr0], _probe_encode(tr0, va0, leaky=True))
print(f"self-test | nested encoding of pure noise: AUC={auc_nested:.5f} (want ~0.5)")
print(f"self-test | leaky  encoding of pure noise: AUC={auc_leaky:.5f} (want clearly above 0.5)")
assert abs(auc_nested - 0.5) < 0.02, "nested encoder is leaking"
assert auc_leaky > 0.55, "leaky control isn't actually leaking, test is broken"
print("self-test passed — nesting removes the row's own label from its own feature.")

With that trusted, encode the two real columns and check what it buys:

In [ ]:
oof_te = np.zeros(len(train))
for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
    te_tr, te_va, _ = nested_target_encode(TE_COLS, tr_idx, va_idx, SEED + 100 + fold)
    Xtr = np.hstack([X_all[tr_idx], te_tr])
    Xva = np.hstack([X_all[va_idx], te_va])
    model = XGBClassifier(n_estimators=2000, max_depth=6, learning_rate=0.05,
                           tree_method="hist", eval_metric="auc",
                           early_stopping_rounds=100, random_state=SEED)
    model.fit(Xtr, y.iloc[tr_idx], eval_set=[(Xva, y.iloc[va_idx])], verbose=False)
    oof_te[va_idx] = model.predict_proba(Xva)[:, 1]

print(f"+ nested target encoding: OOF AUC = {roc_auc_score(y, oof_te):.5f}")
print(f"gain over raw baseline: {roc_auc_score(y, oof_te) - roc_auc_score(y, oof_baseline):+.5f}")

That's the single biggest jump on this page, and it lines up with the mechanism: `Annual_Income_USD` on its own barely registers in a raw model's feature importance, and once it's target-encoded it becomes one of the strongest columns in the whole matrix. The information was always there, the raw column just couldn't carry it to the model.

## 5. Frequency encoding

A second, cheaper signal sits right next to the first one: how often does this exact value occur at all? This reads no labels, so there's no leakage question — the only thing to get right is fitting counts on the training partition only, not on train+test combined. (I checked the transductive version too, counting over train+test together: it's indistinguishable from the train-only version, correlation 0.9996, because train and test are drawn from the same distribution here. Train-only is the more defensible default and costs nothing, so that's what's below.)

In [ ]:
FREQ_COLS = TE_COLS  # only columns with real cardinality benefit from this

def frequency_encode(cols, tr_idx):
    maps = {c: X[c].iloc[tr_idx].value_counts() for c in cols}
    def apply(frame):
        out = np.zeros((len(frame), len(cols)))
        for j, c in enumerate(cols):
            out[:, j] = np.log1p(frame[c].map(maps[c]).fillna(0.0).to_numpy(dtype=np.float64))
        return out
    return apply

oof_freq = np.zeros(len(train))
for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
    te_tr, te_va, _ = nested_target_encode(TE_COLS, tr_idx, va_idx, SEED + 100 + fold)
    freq_apply = frequency_encode(FREQ_COLS, tr_idx)
    Xtr = np.hstack([X_all[tr_idx], te_tr, freq_apply(X.iloc[tr_idx])])
    Xva = np.hstack([X_all[va_idx], te_va, freq_apply(X.iloc[va_idx])])
    model = XGBClassifier(n_estimators=2000, max_depth=6, learning_rate=0.05,
                           tree_method="hist", eval_metric="auc",
                           early_stopping_rounds=100, random_state=SEED)
    model.fit(Xtr, y.iloc[tr_idx], eval_set=[(Xva, y.iloc[va_idx])], verbose=False)
    oof_freq[va_idx] = model.predict_proba(Xva)[:, 1]

print(f"+ frequency encoding: OOF AUC = {roc_auc_score(y, oof_freq):.5f}")

Small but real, and it was worth a submission on its own to confirm CV and the public board agreed (0.94544 OOF → 0.94574 public — CV up, LB up, the relationship you want to see).

## 6. More resolutions: the same idea applied to coarser keys

The exact-value encoding is powerful but each individual value can have thin support. Two things help: encoding the same columns at *coarser* resolutions too (`income // 100`, `income // 1000`, integer-km commute) so nearby values share statistics, and encoding the exact value at more than one smoothing strength so the model can pick whichever version generalises best. I also add nested target encoding for every categorical/low-cardinality column, not just the two big ones — it's cheap and some of them do carry a bit of extra separation once the sparsity is handled the same nested way.

This is the point where I stopped hand-picking single features and started thinking of it as: *give the model several views of the same underlying value and let it choose.*

In [ ]:
DISCRETE_COLS = ["Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
                  "Charging_Stations_Near_Work", "Environmental_Concern_Level"]
KEY_SMOOTHINGS = (1.0, 10.0, 100.0)

def build_key_frame(frame):
    inc = np.rint(frame["Annual_Income_USD"].to_numpy(dtype=np.float64)).astype(np.int64)
    km10 = np.rint(frame["Daily_Commute_km"].to_numpy(dtype=np.float64) * 10).astype(np.int64)
    out = pd.DataFrame({
        "income_floor_100": inc // 100,
        "income_floor_1000": inc // 1000,
        "commute_int": km10 // 10,
    }, index=frame.index)
    for c in cat_cols + DISCRETE_COLS:
        out[f"cat_{c}"] = frame[c].astype(str).to_numpy()
    return out

keys_all = build_key_frame(X)
keys_test = build_key_frame(X_test)

def encode_keys(cols, keys_tr, keys_va, keys_te, y_tr, fold_seed, smoothing):
    n = len(cols)
    out_tr = np.zeros((len(keys_tr), n)); out_va = np.zeros((len(keys_va), n)); out_te = np.zeros((len(keys_te), n))
    for j, col in enumerate(cols):
        v_tr = keys_tr[col]
        sums, counts, prior = fit_target_rate(v_tr, y_tr)
        out_va[:, j] = apply_target_rate(keys_va[col], sums, counts, prior, smoothing)
        out_te[:, j] = apply_target_rate(keys_te[col], sums, counts, prior, smoothing)
        inner = StratifiedKFold(TE_INNER_SPLITS, shuffle=True, random_state=fold_seed)
        for i_tr, i_va in inner.split(np.zeros(len(v_tr)), y_tr):
            s_i, c_i, p_i = fit_target_rate(v_tr.iloc[i_tr], y_tr.iloc[i_tr])
            out_tr[i_va, j] = apply_target_rate(v_tr.iloc[i_va], s_i, c_i, p_i, smoothing)
    return out_tr, out_va, out_te

def lgb_params():
    return dict(objective="binary", n_estimators=20000, learning_rate=0.02,
                num_leaves=32, max_depth=5, min_child_samples=10,
                subsample=0.80, subsample_freq=1, colsample_bytree=0.30,
                reg_alpha=0.071, reg_lambda=2.0, max_bin=1024,
                n_jobs=-1, deterministic=True, force_col_wise=True, verbosity=-1)

def build_full_matrix(tr_idx, va_idx, fold):
    fold_seed = SEED + 100 + fold
    te_tr, te_va, te_te = nested_target_encode(TE_COLS, tr_idx, va_idx, fold_seed)
    y_tr = y.iloc[tr_idx]
    freq_apply = frequency_encode(FREQ_COLS, tr_idx)

    extras_tr, extras_va, extras_te = [te_tr, freq_apply(X.iloc[tr_idx])], [te_va, freq_apply(X.iloc[va_idx])], [te_te, freq_apply(X_test)]
    for m in (1.0, 100.0):  # the exact-value keys again, extra smoothings
        e_tr, e_va, e_te = nested_target_encode(TE_COLS, tr_idx, va_idx, fold_seed, smoothing=m)
        extras_tr.append(e_tr); extras_va.append(e_va); extras_te.append(e_te)
    for m in KEY_SMOOTHINGS:  # the coarser keys and the low-cardinality columns
        k_tr, k_va, k_te = encode_keys(list(keys_all.columns), keys_all.iloc[tr_idx].reset_index(drop=True),
                                        keys_all.iloc[va_idx].reset_index(drop=True), keys_test.reset_index(drop=True),
                                        y_tr.reset_index(drop=True), fold_seed, m)
        extras_tr.append(k_tr); extras_va.append(k_va); extras_te.append(k_te)

    Xtr = np.hstack([X_all[tr_idx]] + extras_tr)
    Xva = np.hstack([X_all[va_idx]] + extras_va)
    Xte = np.hstack([X_test_all] + extras_te)
    return Xtr, Xva, Xte

oof_final = np.zeros(len(train))
pred_final = np.zeros(len(test))
for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
    Xtr, Xva, Xte = build_full_matrix(tr_idx, va_idx, fold)
    model = lgb.LGBMClassifier(**lgb_params(), random_state=SEED + fold)
    model.fit(Xtr, y.iloc[tr_idx], eval_set=[(Xva, y.iloc[va_idx])], eval_metric="auc",
              callbacks=[lgb.early_stopping(500, verbose=False), lgb.log_evaluation(0)])
    oof_final[va_idx] = model.predict_proba(Xva)[:, 1]
    pred_final += model.predict_proba(Xte)[:, 1] / N_SPLITS
    print(f"fold {fold}: AUC = {roc_auc_score(y.iloc[va_idx], oof_final[va_idx]):.5f}")

print(f"\nfull recipe, single 5-fold LightGBM: OOF AUC = {roc_auc_score(y, oof_final):.5f}")

This is the single-model ceiling for this feature set — 0.9461 OOF, 0.94635 on the public board. Everything past this point is either ensembling (real, if small) or a dead end (most of it).

## 7. Model family, and the long list of things that didn't pay off

**LightGBM vs. XGBoost vs. CatBoost, on the exact same matrix above:** all three correlate above 0.998 with each other and land within a few hundredths of a percent of one another. Swapping the boosting library moved the score *less* than changing one target-encoding smoothing constant did. Once the columns are fixed, the optimiser barely matters — I kept LightGBM because it happened to be marginally ahead here, not because I have a principled reason to prefer it.

Everything else on this list is a real, controlled test that came back flat or negative, each checked against a frozen control run on the identical folds so I could be sure a null result was the feature's fault and not a harness bug:

- **Splitting income/commute into individual decimal digits.** A popular trick on the forums, and it's a complete no-op once frequency encoding is already in the matrix — the digits are just a coarser way of expressing the same rarity information the frequency column already gives directly.
- **Bringing in the original 10,000-row source dataset** (novelty flags, per-value frequency from it, competition-vs-source lift ratios). All landed at zero. The source is too small to estimate anything reliable per exact value — most income values in it occur exactly once, so there's nothing to estimate.
- **Income crossed with context** — subsidy, environmental concern, range anxiety, home charging, and separately with commute — as its own nested target encoding, i.e. "does knowing income *and* subsidy together beat knowing them separately." Every single pair came back null or slightly negative, and a GBDT already recombines two marginal encodings via splits, so there's nothing extra to hand it. One of these pairs was worth checking for a second reason: income and commute turn out to be jointly correlated in the raw data itself (the 5 km commute spike is unevenly spread across income bins, a real and fairly strong pattern with no target involved at all) — but that correlation doesn't carry any *target* signal beyond what the two separate per-value encodings already supply. The generator apparently linked those two columns for reasons that have nothing to do with who buys the car.
- **A coarser income view as a second, more decorrelated model to blend against the main one** (dropping the exact-value key entirely, keeping only wide floor bins). It is genuinely less correlated with the main model (~0.9975 instead of >0.999), but it's also just a strictly weaker view of the same information, so the blend gain was too small to be worth carrying a second model.
- **More LightGBM capacity, more regularisation, different subsampling, a slower learning rate.** I swept several variants around my starting parameters; the best of them moved the score by about a tenth of what I'd call a real gain. The parameters I started with (borrowed from a couple of public baselines) were already close to a local optimum for this exact matrix.
- **`id`, and anything derived from it** (mod small numbers, last digits, row position as a fraction of the file). Completely flat — the per-group variation in target rate matches what pure sampling noise alone would produce, almost to the decimal.

None of this is wasted, in the sense that it's the reason I'm confident the recipe above is close to what this feature set can give a tree-based model, rather than one plausible-looking column away from a much better score.

## 8. Final step: bagging away partition noise

The one thing that kept paying, a little, past the point everything else stopped: training the *same* model and features on several different 5-fold splits and averaging. This isn't seed-averaging the model — it's averaging over which rows happened to land in which fold, which turns out to have its own irreducible noise on top of the model's own variance. Three different partitions bought +0.00017 OOF over one partition; going to six bought a bit more, with visibly diminishing returns — the kind of curve you'd expect from averaging down an independent noise term rather than adding information.

In [ ]:
N_PARTITIONS = 6
PARTITION_SEEDS = [SEED + 3 * i for i in range(N_PARTITIONS)]  # e.g. 42, 45, 48, ...

bag_oof = np.zeros(len(train))
bag_pred = np.zeros(len(test))

for p_seed in PARTITION_SEEDS:
    p_folds = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=p_seed).split(X_all, y))
    p_oof = np.zeros(len(train))
    p_pred = np.zeros(len(test))
    for fold, (tr_idx, va_idx) in enumerate(p_folds, start=1):
        Xtr, Xva, Xte = build_full_matrix(tr_idx, va_idx, fold)
        model = lgb.LGBMClassifier(**lgb_params(), random_state=SEED + fold)
        model.fit(Xtr, y.iloc[tr_idx], eval_set=[(Xva, y.iloc[va_idx])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(500, verbose=False), lgb.log_evaluation(0)])
        p_oof[va_idx] = model.predict_proba(Xva)[:, 1]
        p_pred += model.predict_proba(Xte)[:, 1] / N_SPLITS
    print(f"partition seed {p_seed}: OOF AUC = {roc_auc_score(y, p_oof):.5f}")
    bag_oof += p_oof / N_PARTITIONS
    bag_pred += p_pred / N_PARTITIONS

print(f"\n{N_PARTITIONS}-partition bag: OOF AUC = {roc_auc_score(y, bag_oof):.5f}")

## 9. Submission

In [ ]:
submission = sample.copy()
submission[TARGET] = bag_pred
assert submission[ID_COL].equals(sample[ID_COL])
assert submission[TARGET].between(0, 1).all()
submission.to_csv("submission.csv", index=False)
submission.head()

That's the whole pipeline. If you fork this: the frozen fold scheme is `StratifiedKFold(5, shuffle=True, random_state=42)` on file order, so out-of-fold predictions from this notebook are directly poolable with anyone else's who matched it — no need to refit anything to stack against it.

If you find something in the "didn't pay off" section that you think should have worked, that's worth a re-run on a second seed before drawing a conclusion — several of those are close enough to the noise floor that a single measurement isn't the last word.

## Credits

Nothing here is copied from another notebook — every function above is my own — but the recipe itself didn't come out of nowhere, and it's worth naming where the ideas actually came from rather than presenting it as if I'd never read the Discussion or Code tabs:

- The core representation (exact-value nested target encoding + frequency + a GBDT on top of it) converges closely with the public recipes in [najiama's Pure LGBM Model](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637) and [mizushimatoshihiko's Competition-Only LGBM](https://www.kaggle.com/code/mizushimatoshihiko/s6e9-competition-only-lgbm-cv-0-94610-lb-0-94636). I built my own version independently rather than forking either, but reading them first is part of why I knew this representation was worth building at all.
- Encoding the same numeric column at more than one resolution (exact value, then coarser floors) is a binning idea I only encountered second-hand, credited to Markus.JM's CTBoost Astra baseline inside [vinay24baghira's notebook](https://www.kaggle.com/code/vinay24baghira/s6e9-fe-lgbm-0-94637-single-model) — I haven't read Markus.JM's original directly, so that attribution is one step removed and I'd rather say so than not.
- [Дворкин Евгений Владимирович's Single XGB baseline](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94583) was an early public confirmation that frequency encoding alone was worth a measurable amount here, before I'd measured it myself.
- The broader idea that this episode's exploitable signal sits in per-value generator artifacts rather than real-world feature semantics runs through a large share of the competition's Discussion tab — that framing is genuinely collective at this point, not any one post's.

If I've undercredited something specific, I'd rather be told and fix it than have it stand.